# MDS-UPDRS Factor Analysis — Data Preparation

This notebook builds the cleaned, longitudinal MDS-UPDRS (Movement Disorder
Society - Unified Parkinson's Disease Rating Scale) dataset that later
factor-analysis notebooks consume.

**What this notebook does:**
1. Loads the PPMI cohort and restricts to patients with a defined subgroup
   label (idiopathic PD subgroups A/B/C).
2. Loads MDS-UPDRS Parts I, II, and III (both ON- and OFF-medication states).
3. Reshapes each part into a regular patient x visit grid (so every patient
   has one row per visit, even if that visit wasn't actually recorded).
4. Fills in *isolated* missing visits by interpolating from the immediately
   adjacent visits (see `fill_single_visit_gaps` below). Gaps that are more
   than one visit wide are intentionally left as NaN.
5. Pickles each cleaned part to `data/02_processed/PPMI/` for downstream use.

**Output files:**
- `P1_MDSUPDRS.pkl` — Part I (non-motor experiences of daily living)
- `P2_MDSUPDRS.pkl` — Part II (motor experiences of daily living)
- `P3OFF_MDSUPDRS.pkl` — Part III (motor exam), OFF-medication state
- `P3ON_MDSUPDRS.pkl` — Part III (motor exam), ON-medication state


In [24]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import palettable.wesanderson as wes

import seaborn as sns

# Auto-reload local modules (e.g. `distributions`) on change, without
# needing to restart the kernel while iterating.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load Cohort and Subgroup Labels

In [25]:
meta = pd.read_csv('./../../data/01_raw/PPMI/PPMI_META_LINKAGE.csv')

# `subgroup_meta.csv` assigns each idiopathic-PD patient (indexed by PATNO)
# to one of three data-driven subgroups, originally coded as 0 / 1 / 2.
# The three-step replace below swaps codes 0 and 2 (via a temporary 'C'
# placeholder so the two substitutions don't clobber each other), then
# relabels the numeric codes to letters. Net effect: original 0 -> 'C',
# original 1 -> 'B', original 2 -> 'A'.
subgroups = pd.read_csv('./../../data/01_raw/PPMI/subgroup_meta.csv', index_col=0)
subgroups['Group'] = subgroups['Group'].replace(0, 'C')   # temp label to free up 0
subgroups['Group'] = subgroups['Group'].replace(2, 0)     # 2 -> 0
subgroups['Group'] = subgroups['Group'].replace('C', 2)   # temp -> 2 (completes the 0/2 swap)

# Final relabel from swapped numeric codes to letter labels A/B/C.
subgroups['Group'] = subgroups['Group'].replace(0, 'A').replace(1, 'B').replace(2, 'C')

/var/folders/5v/vtmbr3255f59jlqhnxc_l2440000gp/T/ipykernel_52781/1236703805.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  subgroups['Group'] = subgroups['Group'].replace('C', 2)   # temp -> 2 (completes the 0/2 swap)


### Define the Visit Schedule

Event IDs are PPMI's visit codes (e.g. `BL` = baseline, `V02`..`V20` = follow-up visits). We restrict to the subset of visits for which genomic data is available, since downstream analyses join on that visit list.

In [26]:
# Visit codes used throughout this notebook — restricted to the visits for
# which genomic data is available (roughly baseline + every 6 months).
events = ['BL', 'V02', 'V04', 'V05',
          'V06', 'V07', 'V08', 'V09', 'V10',
          'V11', 'V12', 'V13', 'V14', 'V15',
          'V16', 'V17', 'V18', 'V19', 'V20']

# Full PPMI visit vocabulary (including screening 'SC', early-termination
# 'ST', and odd-numbered visits not used here), kept for reference/lookup.
all_events = ['SC', 'BL', 'V01', 'V02', 'V03', 'V04', 'V05',
              'V06', 'V07', 'V08', 'V09', 'V10', 'V11', 'V12',
              'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19',
              'ST', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25']

# Maps each visit in `events` to an approximate elapsed time in years
# (visits are ~6 months apart, i.e. 0.5-year steps), for use in time-based
# modeling downstream.
x_list = pd.DataFrame(np.arange(0, 9.5, 0.5), index=events, columns=['response'])

### Helper: Fill Isolated Missing Visits

Each MDS-UPDRS part is reshaped into a full patient x visit grid, so a
patient who skipped a visit gets a row of NaNs instead of a missing row.
`fill_single_visit_gaps` patches over *isolated* single-visit gaps by
interpolating from the neighboring visits, while leaving wider gaps (2+
consecutive missing visits) as NaN rather than guessing across them.

Filling rule per patient, scanning visits in order:
- Gap with both a valid previous **and** next visit adjacent to it →
  filled with the mean of those two neighbors.
- Gap with only a valid previous visit adjacent → forward-filled.
- Gap with only a valid next visit adjacent → back-filled.
- Consecutive gaps: after the first missing visit is patched, subsequent
  ones in the same run are only back-filled from the next valid visit
  (this is what the `row_na` flag controls).

Note: the very first and last visit for each patient are never filled
(the scan only looks at interior rows), and this modifies a copy of the
input grid.


In [27]:
def fill_single_visit_gaps(df_long, group_col='PATNO', event_col='EVENT_ID'):
    """Fill isolated (single-visit) gaps in a long patient x visit frame.

    Parameters
    ----------
    df_long : DataFrame
        Long-format frame with one row per (patient, visit), where
        `event_col` is an ordered pandas Categorical over the visit codes.
    group_col : str
        Column identifying the patient (default 'PATNO').
    event_col : str
        Column identifying the visit, as an ordered Categorical
        (default 'EVENT_ID').

    Returns
    -------
    DataFrame
        Copy of `df_long` with isolated missing-visit rows interpolated
        from adjacent visits; wider gaps are left as NaN.
    """
    out = df_long.copy()
    for patno in out[group_col].unique():
        row_na = False
        df = out[out[group_col] == patno].copy()
        # Integer position of each visit within the ordered visit list,
        # used to check whether two rows are truly *adjacent* visits
        # (as opposed to adjacent only because of how the frame is sorted).
        df['Temp_ID'] = df[event_col].cat.codes.values

        for i in range(1, len(df) - 1):
            if df.iloc[i].isnull().any():
                prev_row = df.iloc[i - 1]
                next_row = df.iloc[i + 1]

                if row_na:
                    # We're inside a run of consecutive gaps: only look
                    # ahead, so we don't propagate a filled value forward
                    # across more than one real gap.
                    if next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID']:
                        df.iloc[i] = df.iloc[i].fillna(df.iloc[i + 1])
                else:
                    if (prev_row['Temp_ID'] + 1 == df.iloc[i]['Temp_ID']
                            and next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID']):
                        # Isolated single-visit gap: average both neighbors.
                        df.iloc[i] = df.iloc[i].fillna(
                            df.iloc[[i - 1, i + 1]].mean(numeric_only=True)
                        )
                    elif prev_row['Temp_ID'] + 1 == df.iloc[i]['Temp_ID']:
                        # Only the previous visit is adjacent -> forward-fill.
                        df.iloc[i] = df.iloc[i].fillna(df.iloc[i - 1])
                    elif next_row['Temp_ID'] - 1 == df.iloc[i]['Temp_ID']:
                        # Only the next visit is adjacent -> back-fill.
                        df.iloc[i] = df.iloc[i].fillna(df.iloc[i + 1])

                row_na = True
            else:
                row_na = False

        df = df.drop('Temp_ID', axis=1)
        out[out[group_col] == patno] = df.copy()

    return out

## Section 1 — Load MDS-UPDRS Assessments

### Part I — Non-Motor Experiences of Daily Living

In [28]:
# Part I has two source files: a clinician-rated section and a
# patient-questionnaire section. We keep PATNO/EVENT_ID plus any column
# whose name contains 'NP1' (the Part I item codes) from each, then merge
# them into one row per patient-visit.
p1_1 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS-UPDRS_Part_I_10Aug2026.csv')
p1_1_filt = p1_1.loc[:, ['PATNO', 'EVENT_ID'] + [col for col in p1_1.columns if 'NP1' in col]]

p1_2 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS-UPDRS_Part_I_Patient_Questionnaire_10Aug2026.csv')
p1_2_filt = p1_2.loc[:, ['PATNO', 'EVENT_ID'] + [col for col in p1_2.columns if 'NP1' in col]]

p1_final = pd.merge(p1_1_filt, p1_2_filt, on=['PATNO', 'EVENT_ID'])

# Restrict to patients that have a subgroup label (i.e. the idiopathic-PD
# cohort defined earlier).
p1_final = p1_final[p1_final['EVENT_ID'].isin(events)]
p1_final = p1_final[p1_final['PATNO'].isin(subgroups['PATNO'])]

# Combine the clinician-rated (NP1RTOT) and patient-questionnaire
# (NP1PTOT) subtotals into a single Part I total score.
p1_final['NP1TOT'] = p1_final['NP1RTOT'] + p1_final['NP1PTOT']
p1_final = p1_final.drop(['NP1RTOT', 'NP1PTOT'], axis=1)

# PPMI uses the sentinel value 101 to mean "not assessed" — convert to NaN
# before any numeric analysis, and drop exact-duplicate rows.
p1 = p1_final.replace(101, np.nan).drop_duplicates()

In [29]:
p1[~p1['PATNO'].isin(tst)].groupby('EVENT_ID').count()

,PATNO,NP1COG,NP1HALL,NP1DPRS,NP1ANXS,NP1APAT,NP1DDS,NP1SLPN,NP1SLPD,NP1PAIN,NP1URIN,NP1CNST,NP1LTHD,NP1FATG,NP1TOT
EVENT_ID,,,,,,,,,,,,,,,
BL,391,391,391,391,391,391,391,391,391,391,391,391,391,391,391
V02,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263
V04,340,340,340,340,340,340,340,340,340,340,340,340,340,340,340
V05,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347
V06,340,340,340,340,340,340,340,340,340,340,340,340,340,340,340
V07,341,341,341,341,341,341,341,341,341,341,341,341,341,341,341
V08,338,338,338,338,338,338,338,338,338,338,338,338,338,338,338
V09,328,328,328,328,328,328,328,328,328,328,328,328,328,328,328
V10,316,316,316,316,316,316,316,316,316,316,316,316,316,316,316


In [30]:
p1[p1['PATNO'].isin(tst)].groupby('EVENT_ID').count()

,PATNO,NP1COG,NP1HALL,NP1DPRS,NP1ANXS,NP1APAT,NP1DDS,NP1SLPN,NP1SLPD,NP1PAIN,NP1URIN,NP1CNST,NP1LTHD,NP1FATG,NP1TOT
EVENT_ID,,,,,,,,,,,,,,,
BL,144,144,144,144,144,144,141,144,144,144,144,144,144,144,141
V02,92,92,92,92,92,92,89,92,92,92,92,92,92,92,89
V04,122,122,122,122,122,122,121,122,122,122,122,122,122,122,121
V05,80,80,80,80,80,80,80,80,80,80,80,80,80,80,80
V06,115,115,115,115,115,115,115,115,115,115,115,115,115,115,115
V08,120,120,120,120,120,120,120,120,120,120,120,120,120,120,120
V10,103,103,103,103,103,103,103,103,103,103,103,103,103,103,103
V12,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20


In [31]:
tst = p1['PATNO'].unique()[-144:]

In [32]:
with open('./../../data/02_processed/PPMI/P1_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p1, file)

#### Part II — Motor Experiences of Daily Living

In [33]:
# Part II is patient-reported only; keep PATNO/EVENT_ID plus any 'NP2'
# item column, clean the "not assessed" sentinel, and restrict to the
# labeled subgroup cohort.
p2 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS_UPDRS_Part_II__Patient_Questionnaire_10Aug2026.csv')
p2_final = p2.loc[:, ['PATNO', 'EVENT_ID'] + [col for col in p2.columns if 'NP2' in col]]
p2_final = p2_final[p2_final['EVENT_ID'].isin(events)]
p2_final = p2_final.replace(101, np.nan).drop_duplicates()
p2 = p2_final[p2_final['PATNO'].isin(subgroups['PATNO'])]


In [34]:
with open('./../../data/02_processed/PPMI/P2_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p2, file)

#### Part III — Motor Exam, OFF-Medication State

In [35]:
# Part III is the clinician motor exam and is collected in both ON- and
# OFF-medication states (tracked by `PDSTATE`); this block keeps only the
# OFF-state (or unspecified/NaN, i.e. not on a levodopa challenge) rows.
p3 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS-UPDRS_Part_III_10Aug2026.csv')
p3 = p3[p3['PDSTATE'].isin([np.nan, 'OFF'])]

p3.rename(columns = {'HRPOSTMED' : 'LastLevodopa'} , inplace=True)
p3[['PATNO', 'EVENT_ID', 'LastLevodopa']].to_csv('../../data/01_raw/PPMI/last_levodopa_off.csv')

# Keep PATNO/EVENT_ID, Hoehn & Yahr stage (NHY), and any 'NP3' item column.
p3_final = p3.loc[:, ['PATNO', 'EVENT_ID', 'NHY'] + [col for col in p3.columns if 'NP3' in col]]
p3_final = p3_final[p3_final['EVENT_ID'].isin(events)]
p3_final = p3_final.replace(101, np.nan).drop_duplicates()
p3 = p3_final[p3_final['PATNO'].isin(subgroups['PATNO'])]

/var/folders/5v/vtmbr3255f59jlqhnxc_l2440000gp/T/ipykernel_52781/3871473475.py:4: DtypeWarning: Columns (16,21) have mixed types. Specify dtype option on import or set low_memory=False.
  p3 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS-UPDRS_Part_III_10Aug2026.csv')


In [36]:
with open('./../../data/02_processed/PPMI/P3OFF_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p3, file)

### Part III — Motor Exam, ON-Medication State

In [37]:
# Same as the OFF-state block above, but keeping ON-state (or
# unspecified/NaN) rows instead.
p3 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS-UPDRS_Part_III_10Aug2026.csv')
p3 = p3[p3['PDSTATE'].isin([np.nan, 'ON'])]

p3.rename(columns = {'HRPOSTMED' : 'LastLevodopa'} , inplace=True)
p3[['PATNO', 'EVENT_ID', 'LastLevodopa']].to_csv('../../data/01_raw/PPMI/last_levodopa_on.csv')

p3_final = p3.loc[:, ['PATNO', 'EVENT_ID', 'NHY'] + [col for col in p3.columns if 'NP3' in col]]
p3_final = p3_final[p3_final['EVENT_ID'].isin(events)]
p3_final = p3_final.replace(101, np.nan).drop_duplicates()
p3 = p3_final[p3_final['PATNO'].isin(subgroups['PATNO'])]

/var/folders/5v/vtmbr3255f59jlqhnxc_l2440000gp/T/ipykernel_52781/465972178.py:3: DtypeWarning: Columns (16,21) have mixed types. Specify dtype option on import or set low_memory=False.
  p3 = pd.read_csv('./../../data/01_raw/PPMI/MDS-UPDRS/MDS-UPDRS_Part_III_10Aug2026.csv')


In [38]:
with open('./../../data/02_processed/PPMI/P3ON_MDSUPDRS.pkl', 'wb') as file:
    pickle.dump(p3, file)